# Exploratory Data Analysis — Edge AI Acoustic Border Intrusion Detection Dataset

**Paper Title (Draft):** *Edge AI-Based Acoustic Footstep Detection for Border Surveillance Using TinyML and LoRa*

---

### Notebook Objectives
This EDA notebook systematically characterises the `BorderIntrusionDetection-Data` acoustic corpus
prior to TinyML model development. The analysis covers:

| Section | Focus |
|---------|-------|
| §1 | Dataset inventory & file integrity |
| §2 | Class distribution & duration statistics |
| §3 | Waveform morphology per class |
| §4 | Spectral analysis (STFT, Mel, CQT) |
| §5 | MFCC feature characterisation |
| §6 | Handcrafted feature extraction & statistics |
| §7 | Inter-class feature separability (PCA, t-SNE) |
| §8 | Correlation & redundancy analysis |
| §9 | Dataset quality & edge case detection |

**Dataset Path:** `/kaggle/input/datasets/katakuricharlotte/borderintrusiondetection-data/dataset`

In [ ]:
import os, glob, warnings, itertools
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

import librosa
import librosa.display
import soundfile as sf

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from scipy.stats import kurtosis, skew
from scipy.signal import welch

warnings.filterwarnings("ignore")

# ── Research Plot Style ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         12,
    "axes.labelsize":    13,
    "axes.titlesize":    13,
    "axes.linewidth":    1.2,
    "xtick.labelsize":   11,
    "ytick.labelsize":   11,
    "xtick.major.size":  6,
    "ytick.major.size":  6,
    "xtick.minor.size":  3,
    "ytick.minor.size":  3,
    "legend.fontsize":   11,
    "legend.frameon":    True,
    "legend.edgecolor":  "0.4",
    "grid.linestyle":    ":",
    "grid.linewidth":    0.7,
    "grid.alpha":        0.85,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for sp in ax.spines.values():
        sp.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=True, right=True)

CLASS_COLORS = {
    "footsteps": "#2166ac",
    "noise":     "#d6604d",
    "balastic":  "#4dac26",
}
CLASS_MARKERS = {"footsteps": "o", "noise": "s", "balastic": "^"}

DATASET_PATH = Path("/kaggle/input/datasets/katakuricharlotte/borderintrusiondetection-data/dataset")
SR_TARGET    = 16000   # resample target for TinyML pipeline
N_MFCC       = 13
N_FFT        = 512
HOP_LENGTH   = 256
N_MELS       = 40

os.makedirs("figures", exist_ok=True)
print("Classes found:", [p.name for p in sorted(DATASET_PATH.iterdir()) if p.is_dir()])

## §1 — Dataset Inventory & File Integrity

Scan all `.wav` files, extract metadata (duration, sample rate, channels), and
flag any corrupted or unreadable files before proceeding.

In [ ]:
records = []
corrupt = []

for cls_dir in sorted(DATASET_PATH.iterdir()):
    if not cls_dir.is_dir():
        continue
    label = cls_dir.name
    for wav_path in sorted(cls_dir.rglob("*.wav")):
        try:
            info = sf.info(str(wav_path))
            records.append({
                "path":       str(wav_path),
                "filename":   wav_path.name,
                "label":      label,
                "duration_s": round(info.duration, 4),
                "samplerate": info.samplerate,
                "channels":   info.channels,
                "frames":     info.frames,
                "subtype":    info.subtype,
            })
        except Exception as e:
            corrupt.append({"path": str(wav_path), "error": str(e)})

df = pd.DataFrame(records)

print("=" * 55)
print(f"  Total audio files  : {len(df)}")
print(f"  Corrupt / unreadable: {len(corrupt)}")
print(f"  Classes            : {df['label'].unique().tolist()}")
print(f"  Unique sample rates: {df['samplerate'].unique().tolist()}")
print(f"  Unique channels    : {df['channels'].unique().tolist()}")
print("=" * 55)

display(df.groupby("label").agg(
    count=("filename","count"),
    total_duration_s=("duration_s","sum"),
    mean_duration_s=("duration_s","mean"),
    min_duration_s=("duration_s","min"),
    max_duration_s=("duration_s","max"),
    std_duration_s=("duration_s","std"),
).round(3))

if corrupt:
    print("\n⚠ Corrupt files:")
    for c in corrupt:
        print(" ", c)

## §2 — Class Distribution & Duration Analysis

An unbalanced dataset causes biased classifiers. Here we visualise:
- Sample count per class
- Total and per-sample audio duration
- Duration distribution (violin + box overlay)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# ── A: Sample count bar ──────────────────────────────────────────────────────
ax = axes[0]
counts = df["label"].value_counts().reindex(CLASS_COLORS.keys()).dropna()
bars = ax.bar(counts.index, counts.values,
              color=[CLASS_COLORS[c] for c in counts.index],
              edgecolor="black", linewidth=0.8, width=0.55, zorder=3)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            str(int(val)), ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_xlabel("Class")
ax.set_ylabel("Number of Samples")
ax.set_title("(a) Sample Count per Class")
paper_axes(ax)

# ── B: Total duration bar ────────────────────────────────────────────────────
ax = axes[1]
total_dur = df.groupby("label")["duration_s"].sum().reindex(CLASS_COLORS.keys()).dropna() / 60
bars = ax.bar(total_dur.index, total_dur.values,
              color=[CLASS_COLORS[c] for c in total_dur.index],
              edgecolor="black", linewidth=0.8, width=0.55, zorder=3)
for bar, val in zip(bars, total_dur.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}m", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_xlabel("Class")
ax.set_ylabel("Total Duration (minutes)")
ax.set_title("(b) Total Audio Duration per Class")
paper_axes(ax)

# ── C: Duration distribution violin ─────────────────────────────────────────
ax = axes[2]
classes_ordered = [c for c in CLASS_COLORS if c in df["label"].unique()]
data_violin = [df[df["label"] == c]["duration_s"].values for c in classes_ordered]
vp = ax.violinplot(data_violin, positions=range(len(classes_ordered)),
                   showmedians=True, showextrema=True, widths=0.6)
for i, (body, cls) in enumerate(zip(vp["bodies"], classes_ordered)):
    body.set_facecolor(CLASS_COLORS[cls])
    body.set_alpha(0.65)
    body.set_edgecolor("black")
    body.set_linewidth(0.8)
vp["cmedians"].set_color("black")
vp["cmedians"].set_linewidth(1.5)
vp["cbars"].set_linewidth(0.8)
vp["cmins"].set_linewidth(0.8)
vp["cmaxes"].set_linewidth(0.8)

# Overlay box plots
bp = ax.boxplot(data_violin, positions=range(len(classes_ordered)),
                widths=0.12, patch_artist=True,
                medianprops=dict(color="white", linewidth=1.5),
                whiskerprops=dict(linewidth=0.8),
                capprops=dict(linewidth=0.8),
                flierprops=dict(marker="x", markersize=4, markeredgewidth=0.8))
for patch in bp["boxes"]:
    patch.set_facecolor("black")
    patch.set_alpha(0.6)

ax.set_xticks(range(len(classes_ordered)))
ax.set_xticklabels(classes_ordered)
ax.set_xlabel("Class")
ax.set_ylabel("Duration (seconds)")
ax.set_title("(c) Duration Distribution")
paper_axes(ax)

plt.tight_layout()
plt.savefig("figures/fig1_class_distribution.pdf")
plt.show()
print("Saved → figures/fig1_class_distribution.pdf")